### ЗАДАЧА: Пакетная загрузка конфигов деплоя

От DevOps-команды приходит пакет строк с конфигами сервисов для выкладки.
Нужно обработать их так, чтобы:
- валидные конфиги попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, какие сервисы включены по окружениям и какой у них средний timeout.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестное окружение или неправильный флаг включения.


In [7]:
# service|max_retries|timeout_sec|environment|enabled
rows = [
    'auth|3|1.5|prod|on',
    'billing|0|2.0|stage|on',
    'search|two|0.8|dev|off',
    'media|5|-1|prod|on',
    'chat|4|1.2|test|off',
    'mail|2|0.5|stage|maybe',
    'worker|1|3.4|prod|on',
]


class DeployConfigError(Exception):
    pass


class RowFormatError(DeployConfigError):
    pass


class RetriesError(DeployConfigError):
    pass


class TimeoutError(DeployConfigError):
    pass


class EnvironmentError(DeployConfigError):
    pass


class EnabledFlagError(DeployConfigError):
    pass


def parse_config(row):
    
    # TODO: распарсить строку и провалидировать max_retries, timeout_sec, environment, enabled
    
    parts = row.split("|")
    if len(parts) !=5:
        raise RowFormatError ("Ошибка")
    service,max_retries, timeout_sec, environment, enabled = parts
    try:
        max_retries= int(max_retries)
    except ValueError as e:
        raise RetriesError("Ошибка") from e
    if max_retries < 0:
        raise RetriesError("не может быть отрицательным") 
    try:
        timeout_sec = float(timeout_sec)
    except ValueError as e:
        raise TimeoutError("Ошибка") from e
    if timeout_sec < 0:
        raise TimeoutError("не может быть отрицательным")

    valid_environments = {'prod', 'stage', 'dev'}
    if environment not in valid_environments:
        raise EnvironmentError("Неизвестное окружение")
    
    enabled_map = {'on': True, 'off': False}
    if enabled not in enabled_map:
        raise EnabledFlagError("Некорректный enabled")
    enabled = enabled_map[enabled]
    return {
        "service": service,
        "max_retries": max_retries,
        "timeout_sec": timeout_sec,
        "environment": environment,
        "enabled": enabled
    }

    # TODO: при ошибках конвертации использовать raise ... from ...
    # TODO: enabled вернуть как boo
   
# TODO: вызвать load_configs(rows)
# TODO: вывести число валидных конфигов и число ошибок
# TODO: вывести ошибки по типам


def load_configs(rows):

    # TODO: вернуть (configs, errors)
    configs =[]
    errors = []

    for i, row in enumerate(rows):
        try: 
            config = parse_config (row)
            configs.append(config)
        except DeployConfigError as e:
            errors.append({
                'row_index': i,
                'row_content': row,
                'error_type': type(e).__name__,
                'error_message': str(e)
            })

    return configs, errors
configs, errors = load_configs(rows)

print(f"Валидные конфигурации: {len(configs)}")
print(f"Ошибки: {len(errors)}")
if errors:  
    print("Ошибки по типам")
    errors_dict = {}
    for error in errors:
        error_type = error['error_type']
        errors_dict[error_type] = errors_dict.get(error_type, 0) + 1
    
    for error_type, count in errors_dict.items():
        print(f"{error_type}: {count}")
else:
    print("нет Ошибок")
enabled_by_enviroment = {}
for config in configs:
    enabled_by_enviroment.setdefault(config["environment"] , []).append(config)

print("Сервисы по окружениям:")
for env, services in enabled_by_enviroment.items():
    print (f"{env}:{services}")

# TODO: собрать enabled_by_environment: dict[str, list[str]]

# TODO: посчитать average_timeout только по enabled=True
enabled_configs = [c for c in configs if c['enabled']]
if enabled_configs:
    total_timeout = sum(c['timeout_sec'] for c in enabled_configs)
    average_timeout = total_timeout / len(enabled_configs)
    print(f"\nСредний timeout для enabled сервисов: {average_timeout:.2f} сек")
else:
    print("\nНет enabled сервисов для расчёта среднего timeout")


Валидные конфигурации: 3
Ошибки: 4
Ошибки по типам
RetriesError: 1
TimeoutError: 1
EnvironmentError: 1
EnabledFlagError: 1
Сервисы по окружениям:
prod:[{'service': 'auth', 'max_retries': 3, 'timeout_sec': 1.5, 'environment': 'prod', 'enabled': True}, {'service': 'worker', 'max_retries': 1, 'timeout_sec': 3.4, 'environment': 'prod', 'enabled': True}]
stage:[{'service': 'billing', 'max_retries': 0, 'timeout_sec': 2.0, 'environment': 'stage', 'enabled': True}]

Средний timeout для enabled сервисов: 2.30 сек
